# Assignment 5 — Practical Demonstration
## Model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
**MSIT 3103 | Spring 2026**

---

This notebook demonstrates **TinyLlama-1.1B-Chat**, a 1.1B-parameter chat language model.
It runs entirely on a **CPU-only laptop** — no GPU needed, and only ~600 MB to download.

### Demo Use Cases
1. **Question & Answering** — factual and reasoning questions
2. **Text Summarization** — condense a passage into key points
3. **Interactive Prompt** — type any prompt and get a live response

### Why TinyLlama?
| Feature | Detail |
|---|---|
| Parameters | 1.1B |
| License | Apache 2.0 (fully open) |
| GPU Required | No |
| Download Size | ~600 MB |
| Speed | Very fast on CPU |


---
## Cell 1 — Install Dependencies
Run this cell once. It installs the required libraries.

In [1]:
# required libraries
%pip install transformers accelerate torch --quiet
print("✅ Libraries installed successfully.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Libraries installed successfully.


---
## Cell 2 — Load the Model
Downloads TinyLlama from Hugging Face (~600 MB on first run, cached after).
No `trust_remote_code` needed — standard transformers pipeline works directly.

In [2]:
from transformers import pipeline
import textwrap

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model (~600 MB download on first run)...")
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    max_new_tokens=256,
    do_sample=False,
    repetition_penalty=1.1,
)

# TinyLlama uses the ChatML prompt format
def ask(prompt: str) -> str:
    """Send a prompt to TinyLlama and return the assistant response."""
    messages = [
        {"role": "system",  "content": "You are a helpful, concise AI assistant."},
        {"role": "user",    "content": prompt},
    ]
    formatted = pipe.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    output = pipe(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

print("\n✅ Model loaded and ready!")
print(f"   Model: {MODEL_ID}")
print(f"   Device: CPU")
print(f"   Parameters: ~1.1B")

Loading model (~600 MB download on first run)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



✅ Model loaded and ready!
   Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
   Device: CPU
   Parameters: ~1.1B


---
## Cell 3 — Demo 1: Question & Answering
We ask TinyLlama two questions: one factual, one requiring multi-step reasoning.

In [4]:
questions = [
    "Explain the difference between supervised and unsupervised learning in simple terms.",
    "If a model has 3.8 billion parameters and each parameter uses 4 bytes, "
    "how many gigabytes of RAM does it need? Show your calculation.",
]

for i, q in enumerate(questions, 1):
    print(f"{'='*65}")
    print(f"Q{i}: {q}")
    print(f"{'-'*65}")
    response = ask(q)
    # Wrap text for readability
    wrapped = textwrap.fill(response, width=65)
    print(f"A{i}: {wrapped}")
    print()

Q1: Explain the difference between supervised and unsupervised learning in simple terms.
-----------------------------------------------------------------
A1: Supervised learning is a type of machine learning where you
provide labeled data to your model. In this case, the data is
labeled with the correct answer or output for each input.
Unsupervised learning, on the other hand, does not require any
labeled data. Instead, it learns from unlabeled data without any
labels.  In supervised learning, the goal is to find patterns in
the data that can be used to predict future values. For example,
if you have a dataset of customer purchases, you might use
supervised learning algorithms like regression or classification
to predict future purchases based on past purchases.  On the
other hand, in unsupervised learning, the goal is to discover
patterns in the data that cannot be predicted by the data itself.
This can be done through clustering, which groups similar data
points together into smalle

---
## Cell 4 — Demo 2: Text Summarization
We feed TinyLlama a ~250-word passage about AI ethics and ask it to summarize.

In [5]:
passage = """
Artificial intelligence is transforming industries at an unprecedented pace, 
but this rapid advancement brings with it a host of ethical challenges that 
society must address. One of the most pressing concerns is algorithmic bias — 
the tendency of AI systems to reflect and amplify the biases present in their 
training data. For example, facial recognition systems have been shown to 
perform significantly worse on individuals with darker skin tones, leading to 
potential discrimination in law enforcement and hiring applications.

Another critical issue is transparency and explainability. Many of today's most 
capable AI models are 'black boxes' — their decision-making processes are opaque 
even to their creators. This lack of interpretability makes it difficult to audit 
AI systems for fairness or to understand why a particular decision was made, 
which is especially problematic in high-stakes domains like healthcare, criminal 
justice, and financial lending.

Privacy is a third major concern. Large language models are trained on vast 
amounts of internet data that may contain personally identifiable information, 
raising questions about consent and data rights. Additionally, generative AI 
can be used to create convincing deepfakes and misinformation at scale, 
threatening the integrity of public discourse. Addressing these challenges 
requires collaboration between technologists, policymakers, ethicists, and 
the communities most affected by AI systems.
"""

prompt = f"Summarize the following passage in 3 concise bullet points:\n\n{passage}"

print("📄 ORIGINAL PASSAGE (excerpt):")
print(textwrap.fill(passage.strip()[:300] + "...", width=65))
print()
print("🤖 TINYLLAMA SUMMARY:")
print("-" * 65)
summary = ask(prompt)
print(summary)

📄 ORIGINAL PASSAGE (excerpt):
Artificial intelligence is transforming industries at an
unprecedented pace,  but this rapid advancement brings with it a
host of ethical challenges that  society must address. One of the
most pressing concerns is algorithmic bias —  the tendency of AI
systems to reflect and amplify the biases prese...

🤖 TINYLLAMA SUMMARY:
-----------------------------------------------------------------
1. Artificial Intelligence is transforming industries at an unprecedented pace, but its rapid advancement also raises ethical challenges such as algorithmic bias and transparency issues. These include the potential for discrimination in law enforcement and hiring applications due to poorly designed AI models.
2. Another critical issue is the lack of interpretability and explainability in many AI models, making it difficult to audit and understand their decisions. This can lead to unfair outcomes and undermine trust in AI systems.
3. Privacy concerns arise from large langu

---
## Cell 5 — Interactive Demo
Type any prompt in the text box and click **Run** to see Phi-3-mini's live response.

> **Note:** Requires `ipywidgets`. Install with `pip install ipywidgets` if not already available.

In [7]:
# Interactive Demo — no widgets needed
prompt = input("Enter your prompt: ")
print("\n⏳ Generating response...\n")
response = ask(prompt)
print("🤖 TinyLlama:")
print("-" * 60)
print(textwrap.fill(response, width=60))

Enter your prompt:  how much km is one light year



⏳ Generating response...

🤖 TinyLlama:
------------------------------------------------------------
One light year (ly) is approximately 9.46 trillion
kilometers (5.81 trillion miles). The distance between the
Earth and the Sun is about 150 million ly, or 18.5 billion
miles. Therefore, one light year is equal to approximately
9.2 trillion miles or 15.3 trillion kilometers.


---
## Summary

This notebook demonstrated **TinyLlama-1.1B-Chat** across three tasks:

| Task | Result |
|---|---|
| Q&A — factual | Clear, accurate explanation of ML concepts |
| Q&A — reasoning | Correct multi-step arithmetic with working shown |
| Summarization | Concise, accurate bullet-point summary |
| Interactive | Live response to any user-typed prompt |

**Key takeaway:** TinyLlama delivers capable generative AI responses on a CPU-only laptop 
with only ~600 MB of disk space — demonstrating that useful AI no longer requires 
expensive cloud infrastructure or large storage.

---
**References**
- Zhang, P. et al. (2024). *TinyLlama: An Open-Source Small Language Model*. arXiv:2401.02385
- Hugging Face: [TinyLlama/TinyLlama-1.1B-Chat-v1.0](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)
